In [1]:
import pandas as pd
import numpy as np

In [2]:
file_paths = {
    "quarter_1": r"F:\Economic2019\Ookla2019_dataset_1.csv",
    "quarter_2": r"F:\Economic2019\Ookla2019_dataset_2.csv",
    "quarter_3": r"F:\Economic2019\Ookla2019_dataset_3.csv",
    "quarter_4": r"F:\Economic2019\Ookla2019_dataset_4.csv"
}

datasets = {}
for quarter, path in file_paths.items():
    df = pd.read_csv(
        path,
        encoding="latin1",
        engine="python",
        on_bad_lines="skip"
    )
    datasets[quarter] = df

combined_data = []
for quarter_num, df in datasets.items():
    quarter_value = int(quarter_num.split('_')[1])
    df_copy = df.copy()
    df_copy['quarter'] = quarter_value
    df_copy['year'] = 2019
    combined_data.append(df_copy)

aggregated_df = pd.concat(combined_data, ignore_index=True)
print(aggregated_df.shape)


(21891117, 9)


In [3]:
observation_counts = aggregated_df['quarter'].value_counts().sort_index()

for obs_count, count in observation_counts.items():
    print(f"Quadkeys with {obs_count} quarter: {count}")

Quadkeys with 1 quarter: 4877036
Quadkeys with 2 quarter: 4971370
Quadkeys with 3 quarter: 5545233
Quadkeys with 4 quarter: 6497478


In [4]:
ookla_aggregated = aggregated_df.groupby(['quadkey'], as_index=False).agg({
    'avg_d_kbps': 'mean', 
    'avg_u_kbps': 'mean',
    'avg_lat_ms': 'mean',
    'tests': 'sum',
    'devices': 'sum',
    'quarter': 'count',
    'year': 'first',
    'geometry': 'first'
})

ookla_aggregated.rename(columns={'quarter': 'observations'}, inplace=True)

In [5]:
ookla_aggregated.shape

(9059737, 9)

In [6]:
ookla_aggregated.head()

,quadkey,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,observations,year,geometry
0,22133222312322,8200.5,3705.00,53.00,2,2,2,2019,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,22133222330013,9195.0,3347.00,43.00,1,1,1,2019,"POLYGON ((-160.032348632812 70.6399478155463, ..."
2,22133222330023,7592.5,3512.25,44.75,6,5,4,2019,"POLYGON ((-160.043334960938 70.6363054807905, ..."
3,22133222330032,7384.0,3868.00,65.00,1,1,1,2019,"POLYGON ((-160.037841796875 70.6363054807905, ..."
4,22133222330100,8895.0,3429.00,43.00,2,2,1,2019,"POLYGON ((-160.02685546875 70.6417687358462, -..."


In [7]:
observation_counts = ookla_aggregated['observations'].value_counts().sort_index()

for obs_count, count in observation_counts.items():
    print(f"Quadkeys with {obs_count} observations: {count}")

Quadkeys with 1 observations: 3205497
Quadkeys with 2 observations: 1698746
Quadkeys with 3 observations: 1333848
Quadkeys with 4 observations: 2821646


In [8]:
grouped_agg = ookla_aggregated.groupby(['quadkey', 'observations'],  as_index=False).agg({
    'avg_d_kbps': 'mean',      # Speed metrics - use mean
    'avg_u_kbps': 'mean',      # Speed metrics - use mean  
    'avg_lat_ms': 'mean',      # Latency - use mean
    'tests': 'sum',            # Count - use sum
    'devices': 'sum',           # Count - use sum
    'geometry': 'first'    # Keep the geometry from first occurrence
})

print("Aggregated values for each observation count:")
print(grouped_agg)

Aggregated values for each observation count:
                  quadkey  observations  avg_d_kbps  avg_u_kbps  avg_lat_ms  \
0          22133222312322             2      8200.5     3705.00   53.000000   
1          22133222330013             1      9195.0     3347.00   43.000000   
2          22133222330023             4      7592.5     3512.25   44.750000   
3          22133222330032             1      7384.0     3868.00   65.000000   
4          22133222330100             1      8895.0     3429.00   43.000000   
...                   ...           ...         ...         ...         ...   
9059732  3131120230000011             3     16384.0     4677.00   62.333333   
9059733  3131120230010113             2       359.0      494.00   15.000000   
9059734  3131120300000111             3     14579.0     6375.00   45.000000   
9059735  3131120300000113             4      8461.0     4484.75   11.750000   
9059736  3220023001303230             2     10444.0    11761.00   13.000000   

     

In [9]:
df = grouped_agg.copy()

cols_to_scale = ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'tests', 'devices']

for col in cols_to_scale:
    df[col] = df[col] / df['observations']

df['tests'] = df['tests'].round().astype(int)
df['devices'] = df['devices'].round().astype(int)

df = df.drop(columns=['observations', 'quadkey'])


In [10]:
df.head()

,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests,devices,geometry
0,4100.250,1852.5000,26.5000,1,1,"POLYGON ((-160.02685546875 70.6435894914449, -..."
1,9195.000,3347.0000,43.0000,1,1,"POLYGON ((-160.032348632812 70.6399478155463, ..."
2,1898.125,878.0625,11.1875,2,1,"POLYGON ((-160.043334960938 70.6363054807905, ..."
3,7384.000,3868.0000,65.0000,1,1,"POLYGON ((-160.037841796875 70.6363054807905, ..."
4,8895.000,3429.0000,43.0000,2,2,"POLYGON ((-160.02685546875 70.6417687358462, -..."


In [11]:
df.shape

(9059737, 6)

In [12]:
df.to_csv('F:\Economic2019\Ookla_Merged.csv', index=False)

<>:1: SyntaxWarning: invalid escape sequence '\E'
<>:1: SyntaxWarning: invalid escape sequence '\E'
C:\Users\Omar\AppData\Local\Temp\ipykernel_948\530681591.py:1: SyntaxWarning: invalid escape sequence '\E'
  df.to_csv('F:\Economic2019\Ookla_Merged.csv', index=False)
